# 中芯国际（688981.SH）行情分析 Notebook

本 Notebook 可在 Jupyter Lab / Jupyter Notebook 中完整复现以下流程：

1. 通过 Tushare 获取近一年日线数据
2. 数据清洗 + 技术指标计算（MA / RSI / MACD）
3. Plotly 交互式 K 线图 + 成交量图
4. RSI / MACD 指标图
5. 自动技术分析结论与买卖建议

> ⚠️ Tushare token 已内置，如需更换请修改 **Cell 2** 中的 `TUSHARE_TOKEN` 变量
> 📦 依赖：tushare / pandas / numpy / plotly / requests

In [ ]:
# ============================================================
# Cell 1：安装依赖
# ============================================================
import subprocess, sys

packages = [
    "tushare",
    "pandas",
    "numpy",
    "plotly",
    "requests",
]

for pkg in packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        check=False
    )

print("✅ 依赖安装完成")

# ---- 初始化 Plotly 在 Jupyter 中的渲染器 ----
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # Jupyter Lab / Notebook 均适用
print("✅ Plotly 渲染器已初始化")


In [ ]:
# ============================================================
# Cell 2：配置与获取数据
# ============================================================
import requests
import json
import pandas as pd
from datetime import datetime, timedelta

# ---- 配置 ----
TUSHARE_TOKEN = "023457f3e3911d11db046cb10165c91ade16348d70ee76af3102262c"
TS_CODE        = "688981.SH"     # 中芯国际 A 股
DAYS           = 365

# ---- 代理绕过（兼容 Mac 系统代理）----
def fetch_tushare(api_name, params, token=TUSHARE_TOKEN):
    """通过 HTTP 直接调用 Tushare Pro API，绕过本地代理问题"""
    proxies = {"http": None, "https": None}   # 强制不走系统代理
    payload = {
        "api_name": api_name,
        "token":    token,
        "params":   params,
        "fields":   "",
    }
    r = requests.post(
        "http://api.tushare.pro",
        data=json.dumps(payload),
        headers={"Content-Type": "application/json"},
        proxies=proxies,
        timeout=30,
    )
    result = r.json()
    if result.get("code") != 0:
        raise RuntimeError(f"Tushare 错误：{result.get('msg')}")
    cols = result["data"]["fields"]
    data = result["data"]["items"]
    return pd.DataFrame(data, columns=cols)

# ---- 日期范围 ----
end_date   = datetime.today()
start_date = end_date - timedelta(days=DAYS)
start_str  = start_date.strftime("%Y%m%d")
end_str    = end_date.strftime("%Y%m%d")

print(f"获取区间：{start_str} ~ {end_str}")
print(f"股票代码：{TS_CODE}\n")

# ---- 获取日线 ----
df = fetch_tushare(
    api_name="daily",
    params={"ts_code": TS_CODE, "start_date": start_str, "end_date": end_str},
)
print(f"✅ 获取到 {len(df)} 条日线记录")
df.head()


In [ ]:
# ============================================================
# Cell 3：数据清洗与特征工程
# ============================================================
# 类型转换
df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d")
for col in ["open", "high", "low", "close", "pre_close",
            "change", "pct_chg", "vol", "amount"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values("trade_date").reset_index(drop=True)

# ---- 均线 ----
df["ma5"]   = df["close"].rolling(5).mean()
df["ma20"]  = df["close"].rolling(20).mean()
df["ma60"]  = df["close"].rolling(60).mean()

# ---- RSI(14) ----
def calc_rsi(s, period=14):
    delta = s.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss
    return 100 - 100 / (1 + rs)

df["rsi14"] = calc_rsi(df["close"])

# ---- MACD(12,26,9) ----
def calc_macd(s, fast=12, slow=26, signal=9):
    ema_fast = s.ewm(span=fast,  adjust=False).mean()
    ema_slow = s.ewm(span=slow,  adjust=False).mean()
    dif      = ema_fast - ema_slow
    dea      = dif.ewm(span=signal, adjust=False).mean()
    hist     = (dif - dea) * 2
    return dif, dea, hist

df["macd_dif"], df["macd_dea"], df["macd_hist"] = calc_macd(df["close"])

# 保存 CSV
import os
os.makedirs("data", exist_ok=True)
df.to_csv("data/smic_688981_daily.csv", index=False, encoding="utf-8-sig")
print(f"✅ 数据已保存至 data/smic_688981_daily.csv")
print(f"区间：{df['trade_date'].min().date()} ~ {df['trade_date'].max().date()}")
print(f"总记录：{len(df)}\n")
df.tail(3)


In [ ]:
# ============================================================
# Cell 4：K 线图 + 成交量（Plotly 交互式）
# 中国配色：红=涨，绿=跌
# ============================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 涨跌颜色（中国习惯）
df["color"] = df.apply(
    lambda r: "#ef5350" if r["close"] >= r["open"] else "#26a69a",
    axis=1
)

# 子图布局：K 线 + 成交量
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.75, 0.25],
    subplot_titles=(
        f"{TS_CODE}   K线图（近一年）",
        "成交量"
    ),
)

# ---- K 线 ----
fig.add_trace(
    go.Candlestick(
        x=df["trade_date"],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="K线",
        increasing_line_color="#ef5350",   # 红涨
        decreasing_line_color="#26a69a",   # 绿跌
        increasing_fillcolor="rgba(239,83,80,0.7)",
        decreasing_fillcolor="rgba(38,166,154,0.7)",
    ),
    row=1, col=1
)

# ---- 均线 ----
for col, name, clr in [
    ("ma5",  "MA5",  "#f5a623"),
    ("ma20", "MA20", "#e040fb"),
    ("ma60", "MA60", "#00e5ff"),
]:
    fig.add_trace(
        go.Scatter(x=df["trade_date"], y=df[col],
                   mode="lines", name=name, line=dict(color=clr, width=1.2)),
        row=1, col=1
    )

# ---- 成交量柱状图 ----
fig.add_trace(
    go.Bar(
        x=df["trade_date"],
        y=df["vol"],
        name="成交量",
        marker_color=df["color"],
        opacity=0.7,
    ),
    row=2, col=1
)

# ---- 布局 ----
latest = df.iloc[-1]
fig.update_layout(
    title=dict(
        text=(
            f"中芯国际（{TS_CODE}）近一年行情  "
            f"| 最新价：{latest['close']:.2f}  "
            f"涨跌：{latest['pct_chg']:+.2f}%"
        ),
        x=0.02, font_size=16
    ),
    xaxis_rangeslider_visible=False,
    template="plotly_dark",
    height=700,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
)
fig.update_yaxes(title_text="价格（元）", row=1, col=1)
fig.update_yaxes(title_text="成交量（手）", row=2, col=1)

fig.show()


In [ ]:
# ============================================================
# Cell 5：RSI + MACD 技术指标图
# ============================================================
fig2 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.5, 0.5],
    subplot_titles=("RSI(14) 相对强弱指标", "MACD(12,26,9)"),
)

# ---- RSI ----
fig2.add_trace(
    go.Scatter(x=df["trade_date"], y=df["rsi14"],
               mode="lines", name="RSI(14)",
               line=dict(color="#ffa726", width=1.8)),
    row=1, col=1
)
# 超买/超卖参考线
for lvl, clr, label in [(70, "#ef5350", "超买 70"), (30, "#26a69a", "超卖 30")]:
    fig2.add_hline(y=lvl, line_dash="dash", line_color=clr,
                   annotation_text=label, row=1, col=1)

# ---- MACD DIF / DEA ----
fig2.add_trace(
    go.Scatter(x=df["trade_date"], y=df["macd_dif"],
               mode="lines", name="DIF", line=dict(color="#f5a623", width=1.5)),
    row=2, col=1
)
fig2.add_trace(
    go.Scatter(x=df["trade_date"], y=df["macd_dea"],
               mode="lines", name="DEA", line=dict(color="#00e5ff", width=1.5)),
    row=2, col=1
)
# MACD 柱状图
fig2.add_trace(
    go.Bar(
        x=df["trade_date"], y=df["macd_hist"],
        name="MACD Hist",
        marker_color=["#ef5350" if v >= 0 else "#26a69a" for v in df["macd_hist"]],
        opacity=0.6,
    ),
    row=2, col=1
)

fig2.update_layout(
    template="plotly_dark",
    height=550,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
)
fig2.update_yaxes(title_text="RSI", row=1, col=1)
fig2.update_yaxes(title_text="MACD", row=2, col=1)

fig2.show()


In [ ]:
# ============================================================
# Cell 6：技术分析结论 + 综合买卖建议
# ============================================================
from IPython.display import display, Markdown
import pandas as pd

latest   = df.iloc[-1]
prev     = df.iloc[-2]
change   = float(latest["close"]) - float(prev["close"])
pct      = change / float(prev["close"]) * 100
high_52  = df["high"].max()
low_52   = df["low"].min()
avg_vol  = df["vol"].mean()
up_days  = (df["pct_chg"] > 0).sum()
dn_days  = (df["pct_chg"] < 0).sum()
rsi_now  = df["rsi14"].iloc[-1]
macd_now = df["macd_dif"].iloc[-1]
macd_sig = df["macd_dea"].iloc[-1]
pos_pct  = (latest["close"] - low_52) / (high_52 - low_52) * 100
vol_ratio = latest["vol"] / avg_vol

# ---- 均线信号 ----
ma_bull = (
    pd.notna(latest["ma5"]) and pd.notna(latest["ma20"]) and pd.notna(latest["ma60"]) and
    latest["ma5"] > latest["ma20"] > latest["ma60"]
)
ma_bear = (
    pd.notna(latest["ma5"]) and pd.notna(latest["ma20"]) and pd.notna(latest["ma60"]) and
    latest["ma5"] < latest["ma20"] < latest["ma60"]
)
if ma_bull:
    ma_signal = "多头排列（短期 > 中期 > 长期，趋势偏多）"
    ma_lvl    = "偏多"
elif ma_bear:
    ma_signal = "空头排列（短期 < 中期 < 长期，趋势偏空）"
    ma_lvl    = "偏空"
else:
    ma_signal = "均线纠缠，方向不明"
    ma_lvl    = "中性"

# ---- RSI 信号 ----
if pd.notna(rsi_now):
    if rsi_now > 70:
        rsi_sig = f"RSI = {rsi_now:.1f}，进入超买区间，注意回调风险"
        rsi_lvl = "超买"
    elif rsi_now < 30:
        rsi_sig = f"RSI = {rsi_now:.1f}，进入超卖区间，关注反弹机会"
        rsi_lvl = "超卖"
    else:
        rsi_sig = f"RSI = {rsi_now:.1f}，处于正常区间"
        rsi_lvl = "中性"
else:
    rsi_sig = "RSI 数据不足"
    rsi_lvl = "未知"

# ---- MACD 信号 ----
if pd.notna(macd_now) and pd.notna(macd_sig):
    if macd_now > macd_sig and macd_now > 0:
        macd_str = f"DIF({macd_now:.3f}) > DEA({macd_sig:.3f})，金叉且在零轴上方，多头强势"
        macd_lvl = "偏多"
    elif macd_now > macd_sig and macd_now <= 0:
        macd_str = f"DIF({macd_now:.3f}) > DEA({macd_sig:.3f})，金叉但位于零轴下方，反弹力度待观察"
        macd_lvl = "中性偏多"
    elif macd_now < macd_sig and macd_now < 0:
        macd_str = f"DIF({macd_now:.3f}) < DEA({macd_sig:.3f})，死叉且在零轴下方，空头强势"
        macd_lvl = "偏空"
    else:
        macd_str = f"DIF({macd_now:.3f}) < DEA({macd_sig:.3f})，死叉但位于零轴上方，调整力度待观察"
        macd_lvl = "中性偏空"
else:
    macd_str = "MACD 数据不足"
    macd_lvl = "未知"

# ---- 价格位置 ----
if pos_pct > 80:
    pos_str = f"当前价格处于近一年高位（分位 {pos_pct:.1f}%），追高需谨慎"
elif pos_pct < 20:
    pos_str = f"当前价格处于近一年低位（分位 {pos_pct:.1f}%），具备一定安全边际"
else:
    pos_str = f"当前价格处于近一年中部（分位 {pos_pct:.1f}%）"

# ---- 成交量 ----
if vol_ratio > 2:
    vol_str = f"今日成交量放大 {vol_ratio:.1f} 倍，市场关注度高"
elif vol_ratio < 0.5:
    vol_str = f"今日成交量萎缩至均量的 {vol_ratio:.1f} 倍，市场观望"
else:
    vol_str = f"今日成交量约为均量的 {vol_ratio:.1f} 倍，成交正常"

# ---- 综合评分 ----
score = 0
if ma_lvl == "偏多":   score += 2
if ma_lvl == "偏空":   score -= 2
if rsi_lvl == "超买":  score -= 1
if rsi_lvl == "超卖":  score += 1
if macd_lvl == "偏多":       score += 1
if macd_lvl == "中性偏多":   score += 0.5
if macd_lvl == "偏空":       score -= 1
if macd_lvl == "中性偏空":   score -= 0.5
if pos_pct > 80:  score -= 1
if pos_pct < 20:  score += 1
if vol_ratio > 2:  score += 0.5

if score >= 3:
    rec = "买入"
    rec_emoji = "🟢"
elif score >= 1:
    rec = "谨慎买入 / 持有"
    rec_emoji = "🟡"
elif score > -1:
    rec = "观望"
    rec_emoji = "⚪"
elif score > -3:
    rec = "谨慎减仓"
    rec_emoji = "🟠"
else:
    rec = "建议减仓"
    rec_emoji = "🔴"

# ---- 输出报告 ----
ma60_str = f"{latest['ma60']:.2f}" if pd.notna(latest["ma60"]) else "N/A"
arrow   = "📈" if pct >= 0 else "📉"

report = f"""
## 📊 中芯国际（{TS_CODE}）技术分析报告

**数据区间**：{df['trade_date'].min().date()} ~ {df['trade_date'].max().date()}（共 {len(df)} 个交易日）

---

### 📈 行情概览

| 指标 | 数值 |
|------|------|
| 最新价 | **{latest['close']:.2f} 元** |
| 今日涨跌 | {arrow} {change:+.2f} 元（{pct:+.2f}%） |
| 近一年最高 | {high_52:.2f} 元 |
| 近一年最低 | {low_52:.2f} 元 |
| 价格分位 | {pos_pct:.1f}% |
| 今日成交量 | {latest['vol']/1e4:.1f} 万手（均量 {avg_vol/1e4:.1f} 万手）|
| 上涨天数 | {up_days} / {len(df)}（占比 {up_days/len(df)*100:.1f}%）|

---

### 🔍 技术指标分析

**① 均线系统**
> {'🔵' if ma_lvl=='偏多' else '🔴' if ma_lvl=='偏空' else '⚪'} {ma_signal}
> MA5 = {latest['ma5']:.2f}   MA20 = {latest['ma20']:.2f}   MA60 = {ma60_str}

**② RSI(14) 动能**
> {'🔴' if rsi_lvl=='超买' else '🔵' if rsi_lvl=='超卖' else '⚪'} {rsi_sig}

**③ MACD(12,26,9) 趋势**
> {'🔵' if '偏多' in macd_lvl else '🔴' if '偏空' in macd_lvl else '🟡'} {macd_str}

**④ 价格位置**
> {'🔴' if pos_pct>80 else '🔵' if pos_pct<20 else '⚪'} {pos_str}

**⑤ 成交量**
> {'🔵' if vol_ratio>2 else '⚪'} {vol_str}

---

### 🎯 综合操作建议

**{rec_emoji} 综合建议：{rec}**

> ⚠️ **免责声明**：以上分析基于历史价格与技术指标自动生成，
> 仅供参考，不构成任何投资建议。投资有风险，入市需谨慎。
"""

display(Markdown(report))
